In [ ]:
# Analytics Dashboard GUI
# This application provides a simple analytics interface over the SQLite
# retail database. It allows the user to run predefined SQL queries and
# visualise results (e.g., a histogram) directly within a Tkinter window.

import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

DB_PATH = "shop.db"

class AnalyticsDashboard(tk.Tk):
    """
    Tkinter-based analytics dashboard for exploring key metrics
    from the shop database.

    Features:
    - Compute the mean of Avg_Price
    - Plot a histogram of Avg_Price
    - Display the top 5 selling products
    """
    def __init__(self):
        super().__init__()
        self.title("Analytics Dashboard (SQLite)")
        self.geometry("980x620")
        self.resizable(False, False)

        # Open database connection
        self.conn = sqlite3.connect(DB_PATH)
        self.conn.execute("PRAGMA foreign_keys = ON;")
        self.cur = self.conn.cursor()

        

        # Build user interface
        self._build_ui()
        

    def _build_ui(self):
        # Configure a simple custom theme
        style = ttk.Style()
        style.theme_use("clam")  

        style.configure("TFrame", background="#f4f6f8")
        style.configure("TLabelframe", background="#f4f6f8", foreground="#333")
        style.configure("TLabelframe.Label", background="#f4f6f8", foreground="#2c3e50", font=("Segoe UI", 10, "bold"))
        
        style.configure("TButton",
                        font=("Segoe UI", 10),
                        padding=6,
                        background="#3498db",
                        foreground="white")
        
        style.map("TButton",
                  background=[("active", "#2980b9")],
                  foreground=[("active", "white")])
        
        style.configure("Header.TLabel",
                        background="#f4f6f8",
                        foreground="#2c3e50",
                        font=("Segoe UI", 16, "bold"))

        # Header
        header = ttk.Label(self, text="Shop Analytics Dashboard", style="Header.TLabel")
        header.pack(pady=10)

        # Main layout container
        main = ttk.Frame(self)
        main.pack(fill="both", expand=True, padx=12, pady=8)
        main.grid_rowconfigure(0, weight=1)
        main.grid_columnconfigure(1, weight=1)

        # Left panel: query buttons and outputs
        left = ttk.LabelFrame(main, text="Queries", padding=10)
        left.grid(row=0, column=0, sticky="ns", padx=(0, 12))

        ttk.Button(left, text="1) Mean of Avg_Price", width=25, command=self.show_mean).grid(row=0, column=0, padx=10, pady=(12, 8))
        ttk.Button(left, text="2) Histogram of Avg_Price", width=25, command=self.show_hist).grid(row=1, column=0, padx=10, pady=8)
        ttk.Button(left, text="3) Top 5 Selling Items", width=25, command=self.show_top5).grid(row=2, column=0, padx=10, pady=8)
        ttk.Button(left, text="Clear Output", width=25, command=self.clear_output).grid(row=3, column=0, padx=10, pady=(8, 12))

        out_frame = ttk.LabelFrame(left, text="Output")
        out_frame.grid(row=4, column=0, padx=10, pady=(6, 12))

        self.output = tk.Text(out_frame, width=34, height=22)
        self.output.pack(padx=8, pady=8)
        self.output.configure(state="disabled")
        self.configure(background="#f4f6f8")


        # Right panel: histogram plot area
        right = ttk.LabelFrame(main, text="Plot Area", padding=10)
        right.grid(row=0, column=1, sticky="nsew")

        self.fig = plt.Figure(figsize=(6.2, 3.9), dpi=100)
        self.ax = self.fig.add_subplot(111)

        self.canvas = FigureCanvasTkAgg(self.fig, master=right)
        self.canvas.get_tk_widget().pack(fill="both", expand=True, padx=6, pady=6)


        # Footer note
        footer = ttk.Label(self, text="Enjoy the visualization!", font=("Segoe UI", 9))
        footer.pack(pady=(0, 10))

        try:
            ttk.Style().theme_use("clam")
        except:
            pass

    # Helper methods
    
    def write_output(self, text: str):
        self.output.configure(state="normal")
        self.output.insert(tk.END, text + "\n")
        self.output.see(tk.END)
        self.output.configure(state="disabled")

    def clear_output(self):
        self.output.configure(state="normal")
        self.output.delete("1.0", tk.END)
        self.output.configure(state="disabled")

    # Query actions
    def show_mean(self):
        try:
            self.cur.execute("SELECT AVG(Avg_Price) FROM TransactionItem WHERE Avg_Price IS NOT NULL;")
            mean_val = self.cur.fetchone()[0]
            if mean_val is None:
                self.write_output("Mean Avg_Price: No data found.")
            else:
                self.write_output(f"Mean Avg_Price (USD): {mean_val:.2f}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def show_hist(self):
        """Plot a histogram of Avg_Price values."""
        try:
            self.cur.execute("SELECT Avg_Price FROM TransactionItem WHERE Avg_Price IS NOT NULL;")
            prices = [row[0] for row in self.cur.fetchall()]

            if not prices:
                self.write_output("Histogram: No Avg_Price data found.")
                return

            # Clear previous plot and draw new histogram
            self.ax.clear()
            self.ax.hist(prices, bins=30)
            self.ax.set_title("Histogram of Avg_Price")
            self.ax.set_xlabel("Avg_Price (USD)")
            self.ax.set_ylabel("Frequency")
            
            self.fig.subplots_adjust(left=0.10, right=0.97, top=0.90, bottom=0.22)
            self.canvas.draw()

            self.write_output(f"Histogram plotted using {len(prices)} Avg_Price values.")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def show_top5(self):
        """
        Top selling items based on total Quantity sold.
        """
        try:
            self.cur.execute("""
                SELECT
                    p.Product_Description,
                    SUM(COALESCE(ti.Quantity, 0)) AS total_qty
                FROM TransactionItem ti
                JOIN Product p ON p.Product_SKU = ti.Product_SKU
                GROUP BY p.Product_Description
                ORDER BY total_qty DESC
                LIMIT 5;
            """)
            rows = self.cur.fetchall()

            if not rows:
                self.write_output("Top 5: No results.")
                return

            self.write_output("Top 5 Selling Items (by total quantity):")
            for i, (desc, qty) in enumerate(rows, start=1):
                self.write_output(f"{i}. {desc} — {qty:.0f}")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    def on_close(self):
        
        """Safely close the database connection on exit."""
        
        try:
            self.conn.commit()
            self.conn.close()
        except:
            pass
        self.destroy()


if __name__ == "__main__":
    app = AnalyticsDashboard()
    app.protocol("WM_DELETE_WINDOW", app.on_close)
    app.mainloop()
